# SV-ADF Window Explorer

Interactive single-ticker SV-ADF analysis with a configurable window.
Edit the CONFIG cell, run all cells, see the result.

Use this for the meeting with Sarkar — pick any ticker, pick any window,
see the result and the diagnostic plot.

**Erika Francesca Caragnano — Politecnico di Torino**

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110})

from config import settings
from src.io.data        import load_ticker_series
from src.pipelines.svadf import run_one as run_svadf
from src.plotting.diagnostic import plot_svadf_only
from src.plotting.comparison import plot_comparison

## CONFIG — edit and re-run

`WINDOWS` defines which SV-ADF windows to run and overlay on the same panel.
By default both the post-ChatGPT (W1) and pre-ChatGPT (W2) windows are used
so the two periods are directly comparable. Add or remove entries as needed.

In [ ]:
TICKER  = "NVDA"
WINDOWS = [
    (settings.SVADF_DEFAULT_START, settings.SVADF_DEFAULT_END),   # W1: post-ChatGPT
    (settings.SVADF_PRE_GPT_START, settings.SVADF_PRE_GPT_END),   # W2: pre-ChatGPT
]
FORCE   = True   # always recompute when iterating in this notebook

print(f"Ticker  : {TICKER}")
for i, (s, e) in enumerate(WINDOWS, 1):
    print(f"Window W{i}: {s} → {e}")

## Run SV-ADF on both windows

Each window writes its own result file keyed by `(TICKER, start, end)`.
Re-running is safe — existing results are overwritten only when `FORCE=True`.

In [ ]:
series = load_ticker_series(TICKER, settings.FIXED_START, settings.FIXED_END)
print(f"Loaded {TICKER}: T={len(series)}\n")

for i, (w_start, w_end) in enumerate(WINDOWS, 1):
    result = run_svadf(TICKER, series,
                       window_start=w_start, window_end=w_end,
                       force=FORCE)
    ep = result["episode"] if result else None
    if ep is None:
        print(f"W{i} ({w_start}→{w_end}): NO episode detected")
    else:
        print(f"W{i} ({w_start}→{w_end}): Episode {ep['start'][:10]} → "
              f"{ep['end'][:10]} ({ep['duration_days']} days, {ep['collapse_type']})")

## SV-ADF diagnostic plot — both windows on the same panel

In [ ]:
plot_svadf_only(TICKER, windows=WINDOWS);

## GSADF vs SV-ADF comparison — both windows on the same panel

Requires GSADF results on disk (`scripts/03_run_analysis.py --methods gsadf`).

In [ ]:
plot_comparison(TICKER, svadf_windows=WINDOWS);